# OpenRouter (free models) from Google Colab

A minimal question/answer example using the `openai` library against the [OpenRouter](https://openrouter.ai/) OpenAI-compatible API.

**Prerequisites:**
1. Create an API key at https://openrouter.ai/keys (from a machine/network where the site is not blocked).
2. In Colab, open the **🔑 Secrets** panel (key icon, left sidebar) and add a secret named `OPENROUTER_API_KEY` with the key value. Enable *Notebook access*.
3. Run the cells in order.

> Do not paste the key directly into the code: it gets exposed if you share or upload the notebook.

In [ ]:
%pip install -q openai

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    API_KEY = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Outside Colab: use an environment variable or a .env file
    API_KEY = os.environ["OPENROUTER_API_KEY"]

assert API_KEY, "Missing OpenRouter API key"

In [ ]:
from openai import OpenAI

# Models with the ':free' suffix do not consume credit.
# Up-to-date catalogue: https://openrouter.ai/models?q=free
MODEL = "meta-llama/llama-3.3-70b-instruct:free"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence, what is the capital of New Zealand?"},
    ],
)

print(response.choices[0].message.content)

## Variant: streaming response

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what an LLM is in 3 bullet points."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

## List the available free models

In [ ]:
free_models = [m.id for m in client.models.list().data if m.id.endswith(":free")]

print(len(free_models), "free models")
for model_id in sorted(free_models)[:20]:
    print("-", model_id)